In [ ]:
import pandas as pd
import numpy as np
import json
import networkx as nx
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import mutual_info_classif
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import HillClimbSearch, BIC, BayesianEstimator
from pgmpy.inference import VariableElimination

# ------------------ Step 1: Load & Preprocess ------------------

df = pd.read_csv("Discretized_Dataset_with_Context.csv").dropna()

ordinal_mapping = {"low": 0, "medium": 1, "high": 2}
ordinal_columns = [
    "Heart rate", "Systolic Blood Pressure", "Diastolic Blood Pressure",
    "Respiratory rate", "Body temperature", "Pain severity - 0-10 verbal numeric rating [Score] - Reported", "age"
]

for col in ordinal_columns:
    if col in df.columns:
        df[col] = df[col].map(ordinal_mapping)

label_encoders = {}
for col in df.select_dtypes(include="object").columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# ------------------ Step 2: Feature Selection ------------------

X = df.drop(columns=["triage_level"])
y = df["triage_level"]
mi_scores = mutual_info_classif(X, y, discrete_features=True)
mi_df = pd.DataFrame({"Feature": X.columns, "MI": mi_scores})
mi_df = mi_df.sort_values(by="MI", ascending=False)

top_k = 15
initial_top_features = mi_df["Feature"].head(top_k).tolist()

critical_features = [
    "Heart rate", "Systolic Blood Pressure", "Diastolic Blood Pressure",
    "Respiratory rate", "Pain severity - 0-10 verbal numeric rating [Score] - Reported",
    "condition_context", "has_myocardial_infarction", "has_diabetes", "age"
]

# Merge and preserve order
selected_features = list(dict.fromkeys(initial_top_features + critical_features))
selected_columns = selected_features + ["triage_level"]
df = df[selected_columns]

# ------------------ Step 3: Structure Learning ------------------

learned_model = nx.DiGraph()

try:
    print("📈 Starting structure learning...")
    hc = HillClimbSearch(df)
    scoring = BIC(df)
    model_structure = hc.estimate(scoring_method=scoring, max_indegree=2)

    if isinstance(model_structure, nx.DiGraph) and model_structure.edges():
        print(f"✅ Learned structure with {len(model_structure.edges())} edges")
        learned_model = model_structure
    else:
        print("⚠️ Structure learning returned no valid edges, continuing with manual augmentation.")

except Exception as e:
    print(f"❌ Structure learning failed: {e}")

# Ensure all selected features are in the graph
for node in selected_columns:
    learned_model.add_node(node)

# Manually enforce edges to triage_level from key parents
for parent in selected_features:
    if parent != "triage_level":
        learned_model.add_edge(parent, "triage_level")

# Remove cycles if any
while not nx.is_directed_acyclic_graph(learned_model):
    cycle = list(nx.simple_cycles(learned_model))[0]
    learned_model.remove_edge(cycle[0], cycle[1])

print(f"📊 Final DAG has {len(learned_model.nodes())} nodes and {len(learned_model.edges())} edges")

# ------------------ Step 4: Fit Parameters ------------------

bn = DiscreteBayesianNetwork(learned_model.edges())
bn.fit(df, estimator=BayesianEstimator, prior_type="BDeu", equivalent_sample_size=10)

# ------------------ Step 5: Inference ------------------

infer = VariableElimination(bn)

print("\n📍 Nodes in the Bayesian Network:")
print(sorted(bn.nodes()))

desired_evidence = {
    "Heart rate": 2,
    "Systolic Blood Pressure": 1,
    "Diastolic Blood Pressure": 1,
    "Respiratory rate": 2,
    "Pain severity - 0-10 verbal numeric rating [Score] - Reported": 2,
    "condition_context": 0,
    "has_myocardial_infarction": 1,
    "has_diabetes": 1,
    "has_hypertension": 1,  # Uncomment if needed
    "age": 2
}

valid_evidence = {k: v for k, v in desired_evidence.items() if k in bn.nodes()}
missing = [k for k in desired_evidence if k not in valid_evidence]
if missing:
    print(f"⚠️ Skipped evidence keys not in model: {missing}")

print("\n🧪 Evidence used for inference:")
print(valid_evidence)

if "triage_level" in bn.nodes() and valid_evidence:
    result = infer.query(variables=["triage_level"], evidence=valid_evidence)
    print("\n🔍 Inference Result:")
    print(result)
else:
    print("⚠️ Cannot perform inference: 'triage_level' not in model or no valid evidence")

# ------------------ Step 6: Export Model ------------------

model_dict = {
    "nodes": list(bn.nodes()),
    "edges": list(bn.edges()),
    "cpds": [
        {
            "variable": cpd.variable,
            "values": cpd.values.tolist(),
            "evidence": cpd.variables[1:],
            "state_names": {k: list(map(str, v)) for k, v in cpd.state_names.items()}
        } for cpd in bn.get_cpds()
    ]
}

with open("learned_bn_model.json", "w") as f:
    json.dump(model_dict, f, indent=2)

print("\n✅ Bayesian Network trained and exported successfully.")
print("\n📊 Top MI features:")
print(mi_df.head(top_k))


INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'has_hypertension': 'N', 'condition_context': 'N', 'has_stroke': 'N', 'has_sepsis_disorder': 'N', 'has_cardiac_arrest': 'N', 'has_history_of_cardiac_arrest_situation': 'N', 'Systolic Blood Pressure': 'N', 'age': 'N', 'has_history_of_myocardial_infarction_situation': 'N', 'has_myocardial_infarction': 'N', 'has_stress_finding': 'N', 'Body Mass Index': 'N', 'has_not_in_labor_force_finding': 'N', 'has_full-time_employment_finding': 'N', 'Body Weight': 'N', 'Heart rate': 'N', 'Diastolic Blood Pressure': 'N', 'Respiratory rate': 'N', 'Pain severity - 0-10 verbal numeric rating [Score] - Reported': 'N', 'has_diabetes': 'N', 'triage_level': 'N'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'has_hypertension': 'N', 'condition_context': 'N', 'has_stroke': 'N', 'has_sepsis_disorder': 'N', 'has_cardiac_arrest': 'N', 'has_history_of_c

📈 Starting structure learning...


  0%|          | 0/1000000 [00:00<?, ?it/s]

INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'has_hypertension': 'N', 'condition_context': 'N', 'has_stroke': 'N', 'has_sepsis_disorder': 'N', 'has_cardiac_arrest': 'N', 'has_history_of_cardiac_arrest_situation': 'N', 'Systolic Blood Pressure': 'N', 'age': 'N', 'has_history_of_myocardial_infarction_situation': 'N', 'has_myocardial_infarction': 'N', 'has_stress_finding': 'N', 'Body Mass Index': 'N', 'has_not_in_labor_force_finding': 'N', 'has_full-time_employment_finding': 'N', 'Body Weight': 'N', 'Heart rate': 'N', 'Diastolic Blood Pressure': 'N', 'Respiratory rate': 'N', 'Pain severity - 0-10 verbal numeric rating [Score] - Reported': 'N', 'has_diabetes': 'N', 'triage_level': 'N'}


✅ Learned structure with 24 edges
📊 Final DAG has 21 nodes and 30 edges

📍 Nodes in the Bayesian Network:
['Body Mass Index', 'Body Weight', 'Diastolic Blood Pressure', 'Heart rate', 'Pain severity - 0-10 verbal numeric rating [Score] - Reported', 'Respiratory rate', 'Systolic Blood Pressure', 'age', 'condition_context', 'has_cardiac_arrest', 'has_diabetes', 'has_full-time_employment_finding', 'has_history_of_cardiac_arrest_situation', 'has_history_of_myocardial_infarction_situation', 'has_hypertension', 'has_myocardial_infarction', 'has_not_in_labor_force_finding', 'has_sepsis_disorder', 'has_stress_finding', 'has_stroke', 'triage_level']

🧪 Evidence used for inference:
{'Heart rate': 2, 'Systolic Blood Pressure': 1, 'Diastolic Blood Pressure': 1, 'Respiratory rate': 2, 'Pain severity - 0-10 verbal numeric rating [Score] - Reported': 2, 'condition_context': 0, 'has_myocardial_infarction': 1, 'has_diabetes': 1, 'age': 2}

🔍 Inference Result:
+-----------------+---------------------+
| 